### Queue — coffee shop (customers → baristas)

In [ ]:
import asyncio
import random
import time

start_time = time.monotonic()

def log(message: str) -> None:
    elapsed = time.monotonic() - start_time
    print(f"[{elapsed:6.2f}s] {message}")

In [ ]:
async def make_item(item: str = "coffee", brew: float = 2.0, fail: bool = False) -> str:
    try:
        await asyncio.sleep(brew)
    except asyncio.CancelledError:
        log(f"{item}: thrown out")
        raise
    if fail:
        raise RuntimeError(f"out of {item}")
    return item

In [ ]:
NUM_BARISTAS = 3


async def customer_line(order_queue: asyncio.Queue, drinks: list[str]):
    for drink in drinks:
        await asyncio.sleep(random.uniform(0.1, 0.4))  # walking up to order
        await order_queue.put(drink)
        log(f"ordered: {drink}")
    for _ in range(NUM_BARISTAS):
        await order_queue.put(None)


async def barista(order_queue: asyncio.Queue, barista_name: str, served: list):
    while True:
        drink = await order_queue.get()
        if drink is None:
            order_queue.task_done()
            break
        log(f"{barista_name}: making {drink}")
        cup = await make_item(drink, brew=random.uniform(0.2, 0.8))
        served.append(cup)
        order_queue.task_done()

**Run the shop**

In [ ]:
start_time = time.monotonic()

drinks = ["espresso", "latte", "cappuccino", "americano", "mocha", "tea", "espresso", "latte"]
order_queue: asyncio.Queue = asyncio.Queue()
served: list = []

customers = asyncio.create_task(customer_line(order_queue, drinks))
baristas = [
    asyncio.create_task(barista(order_queue, f"barista-{index}", served))
    for index in range(NUM_BARISTAS)
]
await asyncio.gather(customers, *baristas)
log(f"closing time, served {len(served)} drinks")

**Backpressure: counter only fits 2 cups**

Watch the log — `walking up` prints fast, `order accepted` has to wait for the slow barista.

In [ ]:
start_time = time.monotonic()

counter: asyncio.Queue = asyncio.Queue(maxsize=4)


async def fast_customers():
    for index in range(6):
        log(f"customer-{index}: walking up")
        await counter.put(f"drink-{index}")
        log(f"customer-{index}: order accepted -> queue: {list(counter._queue)} (size {counter.qsize()})")


async def slow_barista():
    while True:
        drink = await counter.get()
        if drink is None:
            counter.task_done()
            break
        log(f"barista: took {drink} -> queue now: {list(counter._queue)} (size {counter.qsize()})")
        await make_item(drink, brew=0.5)
        counter.task_done()


customers = asyncio.create_task(fast_customers())
worker = asyncio.create_task(slow_barista())

await customers
await counter.put(None)
await worker

**`PriorityQueue` — VIP first**

In [ ]:
start_time = time.monotonic()

priority_queue: asyncio.PriorityQueue = asyncio.PriorityQueue()

await priority_queue.put((3, "regular tea"))
await priority_queue.put((1, "VIP: triple espresso"))
await priority_queue.put((2, "latte"))

while not priority_queue.empty():
    priority, drink = await priority_queue.get()
    log(f"serving (priority={priority}): {drink}")

---
- Sentinel (`None`) cleanly shuts consumers down.
- `maxsize` gives backpressure; `task_done()` + `join()` wait for completion.
- `PriorityQueue` orders by the first tuple element.